# Aplicación RAG con NVIDIA NIM en Google Colab — versión de David (dguemo)

Copia modificada del notebook `02b` del curso para el **Workshop RAG SI3003** (Bloque A).

| Tarea | Dónde está en este notebook |
|---|---|
| A1 · Ejecutar de punta a punta | Sección 4.1 (corpus demo, 4 preguntas + 1 fuera de contexto) |
| A2 · Corpus propio | Sección 3 (carga `data/company_documents.json`) y 4.2 |
| A3 · Ingeniería del prompt | Sección 4.3 (prompt antes/después) |
| A4 · Score de similitud | `build_pipeline` devuelve el `vectorstore`; sección 4.4 y `chat()` |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

> En Colab, ejecuta las celdas **en orden**. Crea antes el secreto `NVIDIA_API_KEY` (panel 🔑 Secrets) con acceso para el notebook. Las celdas de tareas imprimen tablas Markdown listas para copiar a `RESPUESTAS.md`.

## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.


In [1]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 10.1 MB/s eta 0:00:00


## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. También puedes introducirla de forma oculta cuando la celda la solicite.


In [ ]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
COLLECTION = "company_docs_gradio"

print("✅ Configuración NVIDIA NIM lista")

# ── Elegir un modelo generativo disponible para esta cuenta ──────────────
# Algunas cuentas reciben 404 "Function ... Not found for account" con ciertos modelos
# (reportado en los foros de NVIDIA desde julio de 2026). Se prueba el modelo del curso
# y, si falla, el siguiente de la lista.
from langchain_nvidia_ai_endpoints import ChatNVIDIA

CANDIDATE_MODELS = [
    GENERATION_MODEL,                             # nvidia/nemotron-3-super-120b-a12b (el del curso)
    "nvidia/llama-3.3-nemotron-super-49b-v1.5",
    "nvidia/nemotron-3-nano-30b-a3b",
    "meta/llama-3.3-70b-instruct",
    "meta/llama-3.1-8b-instruct",
]

def pick_generation_model(candidates):
    for model in candidates:
        try:
            ChatNVIDIA(model=model, max_completion_tokens=20).invoke("Responde solo: ok")
            return model
        except Exception as error:
            print(f"⚠️ {model} no disponible: {str(error).splitlines()[0][:120]}")
    raise RuntimeError("Ningún modelo generativo respondió: revisa tu key en build.nvidia.com")

GENERATION_MODEL = pick_generation_model(CANDIDATE_MODELS)
print(f"✅ Modelo generativo: {GENERATION_MODEL}")


import re, time

def with_retry(fn, *args, tries=6, wait=20, **kwargs):
    """Reintenta llamadas a NVIDIA NIM cuando hay rate limit (429)."""
    for attempt in range(1, tries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as error:
            msg = str(error)
            if any(s in msg for s in ("429", "Too Many", "rate limit", "Rate limit")) and attempt < tries:
                print(f"⏳ Rate limit; reintento {attempt}/{tries - 1} en {wait}s")
                time.sleep(wait)
                continue
            raise

## 3. Cargar los documentos

`DEMO_DOCUMENTS` son los 4 documentos del curso (se usan en A1). Mi corpus propio (A2) es **Andina Bikes S.A.S.**, una tienda y taller de bicicletas inventada en Medellín: 10 documentos con metadata `department` y `type`.

La celda busca `data/company_documents.json`; si no existe en la sesión de Colab, lo descarga de mi repositorio público de GitHub. (Corrige la ruta que en el original apuntaba a `Path("company_documents.json")`.)

In [ ]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


# A2: corpus propio en data/company_documents.json (se descarga de mi repo si no está en Colab)
OWN_PATH = Path("data/company_documents.json")
RAW_URL = "https://raw.githubusercontent.com/dguemo/si3003-workshop-rag-dguemo/main/data/company_documents.json"
if not OWN_PATH.exists():
    import urllib.request
    OWN_PATH.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(RAW_URL, OWN_PATH)
        print(f"⬇ Descargado {OWN_PATH} desde GitHub")
    except Exception as error:
        print(f"⚠️ No se pudo descargar el corpus propio: {error}")

demo_documents = DEMO_DOCUMENTS
own_documents = load_documents(OWN_PATH)
company_documents = own_documents
print(f"✅ Demo: {len(demo_documents)} documentos | Propio (Andina Bikes): {len(own_documents)} documentos")

## 4. Construir el pipeline RAG

Separé la construcción en dos funciones para no volver a pagar embeddings cuando solo cambia el prompt o la recuperación:

- `build_index(documents, chunk_size, chunk_overlap)` → fragmenta, calcula embeddings e indexa en Qdrant (inserción en HNSW).
- `build_chain(vectorstore, system_prompt, k, score_threshold)` → retriever + prompt + LLM con LCEL.
- `build_pipeline(...)` combina las dos y **devuelve también el `vectorstore`** (A4).

In [ ]:
PROMPT_ORIGINAL = (
    "Eres un asistente empresarial. Responde únicamente con información "
    "del contexto proporcionado. Si el contexto no contiene la respuesta, "
    "indica que no tienes esa información. Responde en español y de forma concisa."
)

# A3: prompt modificado — cita entre corchetes, frase fija y máximo 3 oraciones
NO_INFO = "No encuentro esa información en los documentos."
PROMPT_A3 = (
    "Eres un asistente empresarial. Responde únicamente con información "
    "del contexto proporcionado. Cita siempre el título del documento entre "
    "corchetes, por ejemplo [Política de reembolsos]. Si el contexto no alcanza, "
    f"responde exactamente: «{NO_INFO}» "
    "Responde en español, en máximo 3 oraciones."
)

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=700,
)
embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
# La dimensión se obtiene del modelo y no queda codificada manualmente.
embedding_dim = len(embeddings.embed_query("prueba de dimensión"))


def format_docs(items: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
        for item in items
    )


def build_index(documents: List[Dict], chunk_size: int = 400, chunk_overlap: int = 40,
                collection: str = COLLECTION):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )
    vectorstore = QdrantVectorStore(client=qdrant_client, collection_name=collection, embedding=embeddings)
    with_retry(vectorstore.add_documents, chunks)
    return vectorstore, chunks


def build_chain(vectorstore, system_prompt: str = PROMPT_ORIGINAL, k: int = 3, score_threshold: float = 0.3):
    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": score_threshold},
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])
    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return setup_and_retrieval.assign(answer=answer_from_documents)


def build_pipeline(documents: List[Dict], system_prompt: str = PROMPT_ORIGINAL,
                   chunk_size: int = 400, chunk_overlap: int = 40,
                   k: int = 3, score_threshold: float = 0.3, collection: str = COLLECTION):
    vectorstore, chunks = build_index(documents, chunk_size, chunk_overlap, collection)
    pipeline = build_chain(vectorstore, system_prompt, k, score_threshold)
    return pipeline, chunks, vectorstore


# ── Utilidades para las tablas de las tareas ──────────────────────────────
REFUSAL = ("no tengo", "no encuentro", "no contiene", "no dispongo", "no se menciona",
           "no hay información", "no cuento", "no se encuentra", "no aparece", "no incluye")

def cell(text, n=220):
    text = re.sub(r"\s+", " ", str(text)).replace("|", "\\|").strip()
    return text if len(text) <= n else text[:n - 1] + "…"

def to_relevance(cos: float) -> float:
    """langchain-qdrant convierte el coseno a 'relevancia' = (coseno + 1) / 2 antes de aplicar score_threshold."""
    return (cos + 1) / 2

def is_refusal(answer: str) -> bool:
    return any(s in answer.lower() for s in REFUSAL)

def auto_check(answer: str, expected) -> str:
    """expected = lista de grupos de palabras (OR dentro del grupo, AND entre grupos) o None si debe negarse."""
    if expected is None:
        return "sí (se negó)" if is_refusal(answer) else "no (respondió algo)"
    low = answer.lower()
    hits = sum(any(w.lower() in low for w in group) for group in expected)
    return "sí" if hits == len(expected) else ("parcial" if hits else "no")

def ask(pipeline, vectorstore, question: str, k: int = 3):
    result = with_retry(pipeline.invoke, question)
    scored = vectorstore.similarity_search_with_score(question, k=k)
    sources = list(dict.fromkeys(d.metadata.get("title", "Sin título") for d in result["documents"]))
    return {"question": question, "answer": result["answer"].strip(), "sources": sources, "scored": scored}

def run_table(pipeline, vectorstore, questions, expected=None, title="Resultados"):
    rows, records = [], []
    header = "| # | Pregunta | Respuesta del RAG | Fuentes (tras umbral) | Top-1 sin umbral (score) |"
    sep = "|---|---|---|---|---|"
    if expected is not None:
        header += " ¿Correcta? (auto) |"; sep += "---|"
    for i, q in enumerate(questions, 1):
        r = ask(pipeline, vectorstore, q)
        top_doc, top_score = r["scored"][0]
        line = (f"| {i} | {cell(q, 90)} | {cell(r['answer'])} | {cell(', '.join(r['sources']) or '—')} "
                f"| {cell(top_doc.metadata.get('title'), 60)} ({top_score:.3f}) |")
        if expected is not None:
            r["check"] = auto_check(r["answer"], expected[i - 1]); line += f" {r['check']} |"
        rows.append(line); records.append(r)
    print(f"### {title}\n"); print(header); print(sep); print("\n".join(rows))
    return records

print(f"✅ Funciones listas (dimensión de embeddings: {embedding_dim})")

### 4.1 · A1 — Corpus de demostración (pipeline original)

Mismos parámetros del notebook del curso: `chunk_size=400`, `chunk_overlap=40`, `k=3`, `score_threshold=0.3` y el prompt original. Se hacen las 4 preguntas de ejemplo y una cuya respuesta **no** está en los documentos.

In [ ]:
print("Construyendo el índice...")
rag_demo, chunks_demo, vs_demo = build_pipeline(demo_documents, collection="demo_docs")
print(f"✅ Pipeline listo: {len(demo_documents)} documentos, {len(chunks_demo)} chunks")

A1_QUESTIONS = [
    "¿Cuál es el precio del plan Enterprise de CloudSync Pro?",
    "¿Cuántos días pueden trabajar remotamente los empleados?",
    "¿Qué pasa con los reembolsos mayores a $100?",
    "¿Qué ocurre durante la primera semana de onboarding?",
    "¿Cuántos días de vacaciones pagadas tienen los empleados al año?",  # fuera de contexto
]
A1_EXPECTED = [
    [["49"], ["anual"]],
    [["3"], ["jefe"]],
    [["supervisor"], ["3 a 5", "3-5"]],
    [["seguridad"], ["cumplimiento"], ["cultura"]],
    None,
]
a1_results = run_table(rag_demo, vs_demo, A1_QUESTIONS, A1_EXPECTED, "A1 · Corpus demo")

### 4.2 · A2 — Mi corpus: Andina Bikes S.A.S.

Primero miro la longitud de los documentos para decidir `chunk_size` (esta celda no llama a la API).

In [ ]:
lengths = [len(d["content"]) for d in own_documents]
print(f"Documentos: {len(own_documents)} | longitud min={min(lengths)}, media={sum(lengths)/len(lengths):.0f}, max={max(lengths)} caracteres")
print("Metadata usada:", sorted({k for d in own_documents for k in d["metadata"]}))
for cs, ov in [(400, 40), (500, 50), (700, 70)]:
    sp = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=ov, separators=["\n\n", "\n", ".", " "])
    parts = [len(sp.split_text(d["content"])) for d in own_documents]
    print(f"chunk_size={cs:>4}, overlap={ov:>3} → {sum(parts):>2} chunks | docs partidos: {sum(p > 1 for p in parts)}")

**Decisión de `chunk_size`:** mis documentos miden entre ~310 y ~570 caracteres y cada uno tiene **dos párrafos** sobre subtemas distintos (p. ej. precios de modelos vs. regalos de compra). Con `chunk_size=700` cada documento queda en un único chunk y el embedding mezcla los dos subtemas; con `400` se parten algunos párrafos a mitad de una oración. Uso **`chunk_size=500`, `chunk_overlap=50` (10 %)**: los documentos largos se parten justo en el salto de párrafo (`\n\n`) y los cortos quedan enteros.

In [ ]:
CHUNK_SIZE_OWN, CHUNK_OVERLAP_OWN = 500, 50

print("Construyendo el índice de Andina Bikes...")
vs_own, chunks_own = build_index(own_documents, CHUNK_SIZE_OWN, CHUNK_OVERLAP_OWN, collection="andina_bikes")
rag_own = build_chain(vs_own, PROMPT_ORIGINAL)
print(f"✅ Pipeline listo: {len(own_documents)} documentos, {len(chunks_own)} chunks")

A2_QUESTIONS = [
    "¿Cuánto cuesta la Palmas E-Bike y qué autonomía tiene?",
    "¿Quién debe aprobar un descuento del 15 %?",
    "¿Qué interés tiene la financiación a 6 cuotas?",
    "¿Cuánto dura la garantía de la batería de la bicicleta eléctrica?",
    "¿Puedo devolver una bicicleta y recibir mi dinero de vuelta?",
    "¿Venden patinetas eléctricas?",            # fuera de contexto
    "¿Cuál es el NIT de Andina Bikes?",          # fuera de contexto
]
A2_EXPECTED = [
    [["6.900.000", "6900000", "6,9"], ["60"]],
    [["gerente comercial"]],
    [["1,5", "1.5"]],
    [["2 años", "dos años"], ["800"]],
    [["no"], ["cambi"]],
    None,
    None,
]
a2_results = run_table(rag_own, vs_own, A2_QUESTIONS, A2_EXPECTED, "A2 · Corpus Andina Bikes")

### 4.3 · A3 — Ingeniería del prompt (antes y después)

Tres preguntas de la tabla A2 (fácil, ambigua y fuera de contexto). Se reutiliza **el mismo índice** y solo cambia el system prompt, así que la diferencia se debe únicamente al prompt.

```diff
- "Eres un asistente empresarial. Responde únicamente con información "
- "del contexto proporcionado. Si el contexto no contiene la respuesta, "
- "indica que no tienes esa información. Responde en español y de forma concisa."
+ "Eres un asistente empresarial. Responde únicamente con información "
+ "del contexto proporcionado. Cita siempre el título del documento entre "
+ "corchetes, por ejemplo [Política de reembolsos]. Si el contexto no alcanza, "
+ "responde exactamente: «No encuentro esa información en los documentos.» "
+ "Responde en español, en máximo 3 oraciones."
```

In [ ]:
A3_QUESTIONS = [
    ("fácil", "¿Qué interés tiene la financiación a 6 cuotas?"),
    ("ambigua", "¿Cuánto cuesta el mantenimiento?"),
    ("fuera de contexto", "¿Venden patinetas eléctricas?"),
]
rag_own_a3 = build_chain(vs_own, PROMPT_A3)

def n_sentences(text):
    return len([s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s])

print("| Tipo | Pregunta | Respuesta ANTES | Respuesta DESPUÉS | ¿Cita [título]? | ¿Frase fija? | Oraciones antes → después |")
print("|---|---|---|---|---|---|---|")
a3_results = []
for kind, q in A3_QUESTIONS:
    before = with_retry(rag_own.invoke, q)["answer"].strip()
    after = with_retry(rag_own_a3.invoke, q)["answer"].strip()
    cites = "sí" if re.search(r"\[[^\]]+\]", after) else "no"
    fixed = "sí" if NO_INFO.lower().rstrip(".") in after.lower() else "no"
    a3_results.append((kind, q, before, after))
    print(f"| {kind} | {cell(q, 60)} | {cell(before)} | {cell(after)} | {cites} | {fixed} | {n_sentences(before)} → {n_sentences(after)} |")

### 4.4 · A4 — Scores de similitud y nuevo `score_threshold`

`vectorstore.similarity_search_with_score(question, k=3)` **no aplica el umbral**: muestra lo que se habría recuperado sin filtrar, que es justo lo que necesito para elegir el umbral con datos. 3 preguntas buenas y 2 fuera de contexto.

In [ ]:
A4_GOOD = A2_QUESTIONS[0:2] + [A2_QUESTIONS[3]]
A4_OUT = [A2_QUESTIONS[5], A2_QUESTIONS[6]]

# Comprobación: ¿en qué escala compara el retriever el score_threshold?
q0 = A4_GOOD[0]
raw = vs_own.similarity_search_with_score(q0, k=1)[0][1]
rel = vs_own.similarity_search_with_relevance_scores(q0, k=1)[0][1]
print(f"Coseno crudo = {raw:.3f} | relevancia usada por score_threshold = {rel:.3f} | (coseno+1)/2 = {to_relevance(raw):.3f}")
print("⚠️ score_threshold=0.3 equivale a coseno ≥ -0.4: en la práctica casi no filtra nada.\n")

print("| Tipo | Pregunta | Fuente 1 (score) | Fuente 2 (score) | Fuente 3 (score) |")
print("|---|---|---|---|---|")
top1 = {"buena": [], "fuera": []}
for kind, qs in (("buena", A4_GOOD), ("fuera", A4_OUT)):
    for q in qs:
        scored = vs_own.similarity_search_with_score(q, k=3)
        top1[kind].append(scored[0][1])
        cols = " | ".join(f"{cell(d.metadata['title'], 45)} ({s:.3f})" for d, s in scored)
        print(f"| {kind} | {cell(q, 70)} | {cols} |")

min_good, max_out = min(top1["buena"]), max(top1["fuera"])
print(f"\nTop-1 mínimo en preguntas buenas: {min_good:.3f}")
print(f"Top-1 máximo en preguntas fuera de contexto: {max_out:.3f}")
if min_good > max_out:
    UMBRAL_A4 = round((min_good + max_out) / 2, 2)
    print(f"➡ Hay separación: umbral propuesto (coseno) = punto medio = {UMBRAL_A4}")
else:
    UMBRAL_A4 = round(max_out, 2)
    print(f"➡ No hay separación limpia: un umbral coseno de {UMBRAL_A4} descartaría alguna pregunta buena; el prompt debe encargarse del resto.")
UMBRAL_A4_RETRIEVER = round(to_relevance(UMBRAL_A4), 3)
print(f"   En el retriever se pasa como score_threshold = (coseno+1)/2 = {UMBRAL_A4_RETRIEVER}")

## 5. Crear la interfaz Gradio

La app permite elegir corpus (**Andina Bikes** con el prompt de A3 y el umbral de A4, o **Demo** con la configuración original) y muestra cada fuente con su **score coseno** (A4). Ojo: `langchain-qdrant` compara `score_threshold` contra `(coseno + 1) / 2`, así que convierto el umbral con `to_relevance()`. ✅ = superó el umbral y llegó al LLM; ⬜ = se habría recuperado sin umbral.

In [ ]:
APPS = {
    "Andina Bikes (propio)": {
        "pipeline": build_chain(vs_own, PROMPT_A3, k=3, score_threshold=UMBRAL_A4_RETRIEVER),
        "vectorstore": vs_own, "threshold": UMBRAL_A4,  # en coseno
        "titles": [d["title"] for d in own_documents], "chunks": len(chunks_own),
    },
    "Demo (CloudSync)": {
        "pipeline": rag_demo, "vectorstore": vs_demo, "threshold": -0.4,  # 0.3 de relevancia = coseno -0.4
        "titles": [d["title"] for d in demo_documents], "chunks": len(chunks_demo),
    },
}


def chat(question: str, history: list | None, corpus: str):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."
    app = APPS[corpus]
    try:
        result = with_retry(app["pipeline"].invoke, question)
        answer = result["answer"].strip()
        # A4: scores sin aplicar el umbral, marcando cuáles sí lo superaron
        scored = app["vectorstore"].similarity_search_with_score(question, k=3)
        sources_md = f"**Fuentes recuperadas (score coseno, umbral coseno = {app['threshold']}):**\n" + "\n".join(
            f"- {'✅' if score >= app['threshold'] else '⬜'} {doc.metadata.get('title', 'Sin título')} — {score:.3f}"
            for doc, score in scored
        )
    except Exception as error:
        answer = "Ocurrió un error al consultar el pipeline. Revisa el detalle mostrado debajo."
        sources_md = f"**Error:** `{type(error).__name__}: {error}`"

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return "", history, sources_md


def corpus_info(corpus: str):
    app = APPS[corpus]
    return ("\n".join(f"- {t}" for t in app["titles"])
            + f"\n\n*{app['chunks']} chunks en el índice · k=3 · umbral coseno={app['threshold']}*")


def clear_chat():
    return [], ""


with gr.Blocks(title="RAG empresarial con NVIDIA NIM") as demo:
    gr.Markdown(
        "# RAG empresarial\n"
        "Consulta documentos internos mediante **LangChain + NVIDIA NIM + Qdrant**."
    )
    with gr.Row():
        with gr.Column(scale=1):
            corpus_radio = gr.Radio(list(APPS), value="Andina Bikes (propio)", label="Corpus")
            gr.Markdown("### Documentos indexados")
            docs_md = gr.Markdown(corpus_info("Andina Bikes (propio)"))
            gr.Markdown("---")
            gr.Markdown(
                f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                f"**LLM:** `{GENERATION_MODEL}`  \n"
                "**Vector store:** Qdrant en memoria (coseno)"
            )
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=430, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(placeholder="Haz una pregunta sobre los documentos...",
                                          label="Pregunta", lines=2, scale=4)
                send_button = gr.Button("Enviar", variant="primary", scale=1)
            sources_box = gr.Markdown()
            clear_button = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[[q] for q in A2_QUESTIONS[:5]] + [[q] for q in A1_QUESTIONS[:4]],
        inputs=question_box,
        label="Preguntas de ejemplo (las 5 primeras son de Andina Bikes; las 4 últimas, del corpus demo)",
    )

    corpus_radio.change(corpus_info, inputs=corpus_radio, outputs=docs_md)
    corpus_radio.change(clear_chat, outputs=[chatbot, sources_box])
    send_button.click(chat, inputs=[question_box, chatbot, corpus_radio],
                      outputs=[question_box, chatbot, sources_box])
    question_box.submit(chat, inputs=[question_box, chatbot, corpus_radio],
                        outputs=[question_box, chatbot, sources_box])
    clear_button.click(clear_chat, outputs=[chatbot, sources_box])

## 6. Ejecutar la aplicación en Colab

`share=True` crea el enlace público temporal. Capturas para el repo:
- `evidencias/A1_gradio.png` → corpus **Demo**, una pregunta de ejemplo con sus fuentes.
- `evidencias/A2_corpus.png` → corpus **Andina Bikes**, una pregunta con respuesta y fuentes.
- `evidencias/A4_scores.png` → panel de fuentes mostrando los scores (✅/⬜).

In [9]:
demo.queue().launch(
    share=True,
    debug=True,
    show_error=True,
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9745d08ed77e63aba6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9745d08ed77e63aba6.gradio.live


### Detener o volver a ejecutar

- Interrumpe la última celda para detener el servidor.
- Antes de volver a ejecutarla, puedes cerrar instancias previas con `gr.close_all()`.
- El enlace público cambia cada vez que se inicia la aplicación.


In [12]:
# Ejecuta esta celda solo cuando quieras cerrar servidores Gradio anteriores.
gr.close_all()
